<h1 style="text-align: center;">Programa 11. Evaluación de modelos de clasificación</h1>
<p style="text-align: center;">24-11-2026</p>
<p style="text-align: center;">
    <img src="img/ITQ.png" width="500">
</p>



**Nombre:** Nikolay Misael De La Cruz Melendres  
**Asignatura:** Fundamentos de Machine Learning  
**Docente:** Elvis David Pachacama Cabezas  
**Repositorio:** [NikolayDlc/fundamentosMachine](https://github.com/NikolayDlc/fundamentosMachine)

**Objetivo.** Aplicar una separación externa *Hold-Out*, realizar una validación cruzada interna estratificada, calcular métricas de clasificación y evaluar finalmente el modelo sobre datos no usados para entrenarlo.

## 1. Importación de las herramientas

- `datasets` permite cargar el conjunto Iris.
- `train_test_split` divide las observaciones en entrenamiento y prueba.
- `StandardScaler` estandariza las características.
- `Pipeline` encadena preprocesamiento y clasificación.
- `SVC` implementa un clasificador de máquinas de vectores de soporte.
- `StratifiedKFold` mantiene aproximadamente la proporción de clases en cada partición; `cross_val_predict` genera predicciones de validación cruzada.
- `sklearn.metrics` calcula exactitud, precisión, sensibilidad (*recall*), F1 y matrices de confusión.

In [1]:
import numpy as np
from pprint import pprint
from sklearn import datasets
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_predict
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.base import clone
from sklearn.svm import SVC
import sklearn.metrics as metrics

## 2. Carga y exploración de los datos

Iris contiene **150 observaciones**, **4 características numéricas** (longitud y anchura de sépalo y pétalo) y **3 clases** de flores: *setosa*, *versicolor* y *virginica*. Como se predice una categoría, se trata de **clasificación**, no de regresión.

En `datos.data` están las características (**X**) y en `datos.target` las etiquetas (**y**).

In [2]:
# Carga de datos, como en el notebook del profesor.
datos = datasets.load_iris()
print('Dimensiones de datos:', np.shape(datos.data))
print('Características:', list(datos.feature_names))
print('Clases:', list(datos.target_names))
print('Cantidad por clase:', dict(zip(datos.target_names, np.bincount(datos.target))))

Dimensiones de datos: (150, 4)
Características: ['sepal length (cm)', 'sepal width (cm)', 'petal length (cm)', 'petal width (cm)']
Clases: [np.str_('setosa'), np.str_('versicolor'), np.str_('virginica')]
Cantidad por clase: {np.str_('setosa'): np.int64(50), np.str_('versicolor'): np.int64(50), np.str_('virginica'): np.int64(50)}


## 3. Separar características y etiquetas

`X` tiene una fila por flor y una columna por característica. `y` contiene la clase correcta de cada fila. Las filas de `X` y `y` deben conservar el mismo orden.

In [3]:
# Extraer la matriz X y el vector de etiquetas y.
X = datos.data
y = datos.target
print('X:', X.shape)
print('y:', y.shape)

X: (150, 4)
y: (150,)


## 4. Partición externa Hold-Out (80 % / 20 %)

El **80 %** se emplea para entrenamiento y validación interna, y el **20 %** se reserva para la evaluación final. `random_state=42` hace reproducible la división.

In [4]:
# Partición externa: no utilizar X_test ni y_test durante la validación interna.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print('Train dimensions:', np.shape(X_train))
print('Test dimensions: ', np.shape(X_test))
print('Clases en train:', np.bincount(y_train))
print('Clases en test: ', np.bincount(y_test))

Train dimensions: (120, 4)
Test dimensions:  (30, 4)
Clases en train: [40 40 40]
Clases en test:  [10 10 10]


## 5. Estandarización y creación del clasificador SVC

En el archivo original se estandarizaba `X_train` antes de la validación cruzada y, además, se creaba un `Pipeline` que luego se sobrescribía con `SVC`. La corrección es trabajar **directamente con X_train y X_test originales** y colocar `StandardScaler` **dentro de Pipeline**. Así, al entrenar el `Pipeline` sobre cada fold, el escalador aprende solo de su parte de entrenamiento.

**Parámetros del ejercicio:** `C=0.1`, `gamma='auto'`, `kernel='rbf'`. La cadena `gamma='autos'` es inválida. `C` controla el compromiso entre regularización y errores, `gamma` influye en el alcance del kernel RBF, y `rbf` modela fronteras no lineales.

**Importante:** construir `alg` no lo entrena todavía.

In [5]:
# Algoritmo compuesto: primero estandarizar y después clasificar.
alg = Pipeline([
    ('scaler', StandardScaler()),
    ('svc', SVC(C=0.1, gamma='auto', kernel='rbf'))
])
print(alg)

Pipeline(steps=[('scaler', StandardScaler()),
                ('svc', SVC(C=0.1, gamma='auto'))])


## 6. Validación cruzada interna con StratifiedKFold

`StratifiedKFold(n_splits=5, shuffle=True, random_state=42)` genera cinco particiones manteniendo aproximadamente la proporción de cada especie. En cada iteración se entrena con cuatro partes y se predice la quinta.

`cross_val_predict(alg, X_train, y_train, cv=cv)` devuelve **una predicción fuera de fold para cada muestra de entrenamiento**. Por tanto, `y_pred_cv` y `y_train` pueden compararse para crear la matriz de confusión y el informe de métricas. No son predicciones del conjunto externo de prueba.

In [6]:
# Validación cruzada interna: predicciones fuera del fold de entrenamiento.
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
y_pred_cv = cross_val_predict(alg, X_train, y_train, cv=cv)
print('Número de predicciones de validación:', len(y_pred_cv))
print('Primeras predicciones:', y_pred_cv[:15])
print('Primeras etiquetas reales:', y_train[:15])

Número de predicciones de validación: 120
Primeras predicciones: [0 1 2 0 1 2 1 1 2 2 2 1 1 1 1]
Primeras etiquetas reales: [0 2 1 0 1 2 1 2 2 2 2 1 1 1 1]


## 7. Métricas de clasificación en validación cruzada

- **Exactitud (accuracy):** proporción total de predicciones correctas.
- **Precisión (precision):** de las observaciones predichas como una clase, qué proporción pertenece realmente a ella.
- **Sensibilidad (recall):** de las observaciones que realmente pertenecen a una clase, qué proporción se detectó.
- **F1:** media armónica de precisión y sensibilidad.

`average='macro'` calcula la métrica por clase y después realiza un promedio simple de las tres clases. Se conserva el enfoque de evaluación del profesor.

In [7]:
# Métricas sobre predicciones de validación cruzada.
print('Exactitud: %.3f' % metrics.accuracy_score(y_train, y_pred_cv))
print('Precisión: %.3f' % metrics.precision_score(y_train, y_pred_cv, average='macro', zero_division=0))
print('Sensibilidad: %.3f' % metrics.recall_score(y_train, y_pred_cv, average='macro', zero_division=0))
print('F1-score: %.3f' % metrics.f1_score(y_train, y_pred_cv, average='macro', zero_division=0))

Exactitud: 0.892
Precisión: 0.899
Sensibilidad: 0.892
F1-score: 0.891


## 8. Matriz de confusión e informe de clasificación

En la **matriz de confusión**, las filas corresponden a las clases reales y las columnas a las clases predichas. La diagonal contiene los aciertos; los valores fuera de ella representan confusiones entre especies.

`classification_report` muestra precisión, sensibilidad, F1 y `support` (cantidad de muestras) por especie.

In [8]:
# Matriz e informe de la validación cruzada interna.
print('Orden de clases:', list(datos.target_names))
print('Matriz de confusión (validación cruzada):\n',
      metrics.confusion_matrix(y_train, y_pred_cv))
print('\nTabla de métricas (validación cruzada):\n',
      metrics.classification_report(
          y_train, y_pred_cv,
          target_names=datos.target_names,
          zero_division=0
      ))

Orden de clases: [np.str_('setosa'), np.str_('versicolor'), np.str_('virginica')]
Matriz de confusión (validación cruzada):
 [[40  0  0]
 [ 0 37  3]
 [ 0 10 30]]

Tabla de métricas (validación cruzada):
               precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        40
  versicolor       0.79      0.93      0.85        40
   virginica       0.91      0.75      0.82        40

    accuracy                           0.89       120
   macro avg       0.90      0.89      0.89       120
weighted avg       0.90      0.89      0.89       120



## 9. Entrenar el modelo final con todo el entrenamiento externo

`clone(alg)` construye una copia **sin entrenar** del `Pipeline` con los mismos parámetros. `fit(X_train, y_train)` ajusta el escalador y el SVC usando las 120 observaciones externas de entrenamiento.

In [9]:
# Resolver los ??? originales: definir, entrenar y evaluar el modelo final.
model = clone(alg)
model = model.fit(X_train, y_train)

# score() en clasificadores SVC devuelve exactitud sobre test.
test_results = model.score(X_test, y_test)
print('Exactitud en test:', np.round(test_results * 100, 4), '%')

Exactitud en test: 90.0 %


## 10. Predicciones y métricas en el conjunto externo de test

El test contiene 30 flores que quedaron fuera de la validación cruzada. `model.predict(X_test)` obtiene sus clases predichas, y después se comparan con `y_test` para calcular las métricas finales. Mantén siempre el mismo orden: **valor real primero, predicción después** al usar `classification_report` y `confusion_matrix`.

In [10]:
# Resolver los ??? originales: predicciones e informe final del test.
y_pred_test = model.predict(X_test)
print('Predicciones del test:', y_pred_test)
print('Valores reales del test:', y_test)
print('\nTabla de métricas (test):\n',
      metrics.classification_report(
          y_test, y_pred_test,
          target_names=datos.target_names,
          zero_division=0
      ))
print('Matriz de confusión (test):\n',
      metrics.confusion_matrix(y_test, y_pred_test))

Predicciones del test: [0 2 1 1 0 1 0 0 2 1 2 2 2 1 0 0 0 1 1 1 0 2 1 1 2 2 1 0 2 0]
Valores reales del test: [0 2 1 1 0 1 0 0 2 1 2 2 2 1 0 0 0 1 1 2 0 2 1 2 2 1 1 0 2 0]

Tabla de métricas (test):
               precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        10
  versicolor       0.82      0.90      0.86        10
   virginica       0.89      0.80      0.84        10

    accuracy                           0.90        30
   macro avg       0.90      0.90      0.90        30
weighted avg       0.90      0.90      0.90        30

Matriz de confusión (test):
 [[10  0  0]
 [ 0  9  1]
 [ 0  2  8]]


## 11. Interpretación y conclusiones

1. La **validación cruzada interna** estima el comportamiento del procedimiento de entrenamiento sobre las 120 muestras de desarrollo; sus predicciones se generan fuera del fold correspondiente.
2. La **evaluación externa** usa las 30 observaciones reservadas y constituye la comprobación final de esta división Hold-Out.
3. La matriz de confusión identifica qué especies se confunden, mientras que exactitud, precisión, sensibilidad y F1 resumen el rendimiento.
4. La estandarización se realiza dentro de `Pipeline`, lo que evita ajustar el escalador con las observaciones del fold de validación.
5. Las métricas obtenidas corresponden a esta partición y a esta configuración de SVC. **No se llevó a cabo optimización de hiperparámetros ni validación clínica.**